# 1. Import libraries

In [1]:
import pandas as pd
import numpy as np

# 2. Load data and get overview (intuitively)

In [2]:
# Load data from CSV file
nf = pd.read_csv('D:/AIO2025/SQL_Project/netflix-data-engineering-project/data/raw/netflix_titles.csv')
nf.head(5)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [3]:
# Characteristic of dataset
nf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


* A preliminary view manually based on overview netflix dataset:
* Netflix dataset: 12 colunms and 8807 rows
* 6 columns contain null value: 
    * Director 
    * cast
    * country
    * date_added
    * rating
    * duration 
 => Handle null values on these columns
 ---
 * Data type: Some have inaccurate type, some have multi-value, insconsistent data
    * date_added is not date type => convert 
    * duration: inconsistent, many kinds of unit: season, min
    * listed: multi-value => split column as needed

# 2. Automatically profiling
This is a frame based on 5 Data Quality Dimensions standard on data engineering, applies to Netflix dataset. Each dimensions will answer a question.
1. Completeness - Is the data incompleted?
2. Uniqueness - Is the data duplicated?
3. Validity - Is the data formated accuratedly?
4. Consistency - Is the data inconsistent?
5. Accuracy/Plausibility - Is the data illogical?

In [4]:
# 1. check completeness function
def check_completeness(df: pd.DataFrame) -> pd.DataFrame:
    missing = df.isnull().sum()
    pct=(missing/len(df)*100).round(2)
    result = pd.DataFrame({'missing_count':missing, 'missing_pct': pct})
    return result[result['missing_count']>0].sort_values('missing_pct', ascending=False)

In [5]:
check_completeness(nf)

,missing_count,missing_pct
director,2634,29.91
country,831,9.44
cast,825,9.37
date_added,10,0.11
rating,4,0.05
duration,3,0.03


In [6]:
# 2. Check uniqueness function
def check_uniqueness(df: pd.DataFrame, key_col: str) -> None:
    print(f"Numbers of compteley duplicated rows: {df.duplicated().sum()}")
    print(f"Number of dupicated '{key_col}': {df[key_col].duplicated().sum()}")
    dup_keys = df[df[key_col].duplicated(keep=False)]
    if len(dup_keys) > 0:
        print(f"Warning: {len(dup_keys)} rows have {key_col} duplicated")
# Need to check show_id because show_id will be the primary key to distinguish rows
check_uniqueness(nf,'show_id')


Numbers of compteley duplicated rows: 0
Number of dupicated 'show_id': 0


In [7]:
# 3. Check_validity function
def check_validity(df:pd.DataFrame) -> None:
    print(df.dtypes)
    
    print("\n--- Scan all columns to detect date-like columns---")
    for col in df.select_dtypes(include='object').columns:
        parsed = pd.to_datetime(df[col], errors='coerce', format='mixed')
        success_rate = parsed.notna().mean()
        if success_rate > 0.8:
            sample = df[col].dropna().iloc[0]
            print(f"{col}: {success_rate:0.0%} parsed as date (sample: '{sample}') needs conversion")
check_validity(nf)

show_id         object
type            object
title           object
director        object
cast            object
country         object
date_added      object
release_year     int64
rating          object
duration        object
listed_in       object
description     object
dtype: object

--- Scan all columns to detect date-like columns---
date_added: 100% parsed as date (sample: 'September 25, 2021') needs conversion


In [8]:
# 4. Check consistency function
def check_consistency(df:pd.DataFrame, col:str, top_n:int = 20) -> None:
    print(f"--- {col}: {df[col].nunique()} unique value---")
    print(df[col].value_counts().head(top_n))
for col in nf.columns:
    check_consistency(nf, col)
    print()
# Result:
#director, cast, country, listed_in contain multi-value

--- show_id: 8807 unique value---
show_id
s1       1
s5875    1
s5869    1
s5870    1
s5871    1
s5872    1
s5873    1
s5874    1
s5876    1
s5850    1
s5877    1
s5878    1
s5879    1
s5880    1
s5881    1
s5882    1
s5868    1
s5867    1
s5866    1
s5865    1
Name: count, dtype: int64

--- type: 2 unique value---
type
Movie      6131
TV Show    2676
Name: count, dtype: int64

--- title: 8807 unique value---
title
Dick Johnson Is Dead                                             1
Ip Man 2                                                         1
Hannibal Buress: Comedy Camisado                                 1
Turbo FAST                                                       1
Masha's Tales                                                    1
Chelsea Does                                                     1
Ricardo O'Farrill Abrazo Genial                                  1
Ip Man                                                           1
Tom Segura: Mostly Stories                   

In [9]:

def detect_multivalue_columns(df: pd.DataFrame, delimiter: str =',',threshold: float =0.1) -> list:
    candidates = []
    for col in df.select_dtypes(include = 'object').columns:
        contains_delim = df[col].dropna().astype(str).str.contains(delimiter).mean()
        if contains_delim > threshold:
            candidates.append(col)
            print(f"{col}:{contains_delim:.0%} values contain '{delimiter}' -> multi-value")
    return candidates
multi_value_col = detect_multivalue_columns(nf)
multi_value_col
# /*This function is trying to detect columns contain multi-values based on delimiter =',', but it also returns some
# columns that are not really multi-value like date_added, description
# => Design another function to scan and return 3 parameters pct_contain_delimiter, average length, average parts */

cast:89% values contain ',' -> multi-value
country:17% values contain ',' -> multi-value
date_added:100% values contain ',' -> multi-value
listed_in:77% values contain ',' -> multi-value
description:73% values contain ',' -> multi-value


['cast', 'country', 'date_added', 'listed_in', 'description']

In [10]:
def scan_delimiter_stats(df: pd.DataFrame, delimiter: str = ',') ->pd.DataFrame:
    stats = []
    for col in df.select_dtypes(include='object').columns:
        non_null = df[col].dropna().astype(str)
        pct = non_null.str.contains(delimiter).mean()
        avg_len = non_null.str.len().mean()
        avg_parts = non_null.str.split(delimiter).str.len().mean()
        stats.append({
            'column':col,
            f'pct_contains_{delimiter}': round(pct,2),
            'avg_length': round(avg_len, 1),
            'avg_parts': round(avg_parts, 1)
        })
    return pd.DataFrame(stats).sort_values(f'pct_contains_{delimiter}', ascending=False)

scan_delimiter_stats(nf)

,column,"pct_contains_,",avg_length,avg_parts
6,date_added,1.00,14.7,2.0
4,cast,0.89,119.8,8.0
9,listed_in,0.77,33.4,2.2
10,description,0.73,143.3,2.0
5,country,0.17,12.6,1.3
3,director,0.10,15.4,1.1
2,title,0.02,17.7,1.0
0,show_id,0.00,4.9,1.0
1,type,0.00,5.6,1.0
7,rating,0.00,4.4,1.0



| Column | Comment | Multi-value or not |
|---|---|---|
|date_added | fixed formart Month date, year, always contains "," | Not |
|cast | multi-value contains many specific names but some rows have only name | multi-value |
|listed_in |the average part is quite small, contains some kinds of genre | multi-value|
|description | absolutely not true multi-value, they are comments | Not|
|country | several values contain more than 1 country | multi-value|
|director |the same as cast|multi-value|
|title | | not|
|show_id | | not|
|type| | not|
|rating | | not|
|duration | | not|

In [11]:
# 5 Check Plausibility function
def check_plausibility(df: pd.DataFrame) -> None:
    current_year = pd.Timestamp.now().year
    invalid = df[(df['release_year'] <1900) | (df['release_year'] > current_year)]
    print(f'Illogical release_year: {len(invalid)} rows')
    print(df['release_year'].describe())

check_plausibility(nf)

Illogical release_year: 0 rows
count    8807.000000
mean     2014.180198
std         8.819312
min      1925.000000
25%      2013.000000
50%      2017.000000
75%      2019.000000
max      2021.000000
Name: release_year, dtype: float64


Cross- field consitency
Check `date_added` must be after `release_year`

In [15]:
nf['date_added_parsed'] = pd.to_datetime(nf['date_added'].str.strip(), format='mixed', errors='coerce')
invalid = nf[nf['date_added_parsed'].dt.year < nf['release_year']]
print(f"Number of rows where date_added is earlier than release_year: {len(invalid)}")
invalid[['title', 'release_year', 'date_added']].head(len(invalid))

Number of rows where date_added is earlier than release_year: 14


,title,release_year,date_added
1551,Hilda,2021,"December 14, 2020"
1696,Polly Pocket,2021,"November 15, 2020"
2920,Love Is Blind,2021,"February 13, 2020"
3168,Fuller House,2020,"December 6, 2019"
3287,Maradona in Mexico,2020,"November 13, 2019"
3369,BoJack Horseman,2020,"October 25, 2019"
3433,The Hook Up Plan,2020,"October 11, 2019"
4844,Unbreakable Kimmy Schmidt,2019,"May 30, 2018"
4845,Arrested Development,2019,"May 29, 2018"
5394,Hans Teeuwen: Real Rancour,2018,"July 1, 2017"


In [16]:
nf.loc[invalid.index, 'type'].value_counts()

type
TV Show    12
Movie       2
Name: count, dtype: int64

# Summary profiling process


|Columns|Issues|Recommendation|
|---|---|---|
|`director`|29.91% missing value + weak multi-value|Keep null (not fill), split the first director if need to simplify|
|`country`|9.44% missing value + multi-value|keep null when aggregateing, split table if analyze by country|
|`cast`|9.37% missing value + strong multi-value|split to `cast` table by many to many|
|`date_added`|0.11% missing, string has blank space|`.str.strip()` +`pd.to_datetime(format='mixed')`|
|`rating`,`duration`|missing insignificantly|fill default value or drop rows|
|`duration`|inconsistent unit: min/ season based on `type`|split `duration_minutes`, `duration_seasons`|
|`listed_in`|multi-value|Split to `genres` table|
|`date_added` vs `released_yrea`|14 rows have `date_added` earlier than `released_year`, most are TV Show|Not modify. Assumption: `release_year` = year of the latest season|